In [1]:
# RAG Evaluation: Precision, Recall, MRR
# Import all required libraries
import os
import warnings
from pathlib import Path
from typing import List, Dict, Tuple, Set
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain.schema import Document

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Imports successful!")

c:\Users\heman\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports successful!


In [2]:
# Load embedding model (same as chatbot)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# Load ChromaDB vector store
CHROMA_DIR = "./chroma_db"

if not os.path.exists(CHROMA_DIR):
    print("⚠️ ChromaDB not found! Run: streamlit run app.py → Process a repository first")
else:
    vectorstore = Chroma(persist_directory=CHROMA_DIR, embedding_function=embeddings)
    total_docs = vectorstore._collection.count()
    print(f"✅ Loaded ChromaDB with {total_docs} documents")
    
    # Create retriever (k=6 same as chatbot)
    retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 6})
    print("✅ Retriever initialized (k=6 top documents)")

✅ Loaded ChromaDB with 186 documents
✅ Retriever initialized (k=6 top documents)


In [3]:
# View sample documents to understand repository structure
sample_docs = vectorstore.similarity_search("function component", k=10)

print("📄 Sample Documents in Vector Store:")
print("=" * 80)

for i, doc in enumerate(sample_docs[:5], 1):
    source = doc.metadata.get('source', 'Unknown')
    print(f"\n{i}. Source: {source}")
    print(f"   Content preview: {doc.page_content[:150]}...")
    print(f"   Length: {len(doc.page_content)} characters")

📄 Sample Documents in Vector Store:

1. Source: cloned_repo\client\src\pages\recruiter\ManageJobs.jsx
   Content preview: import React, { useState, useEffect } from "react";
import { AnimatePresence } from "framer-motion";
import {
    Edit3,
    Trash2,
    Eye,
    Cale...
   Length: 38841 characters

2. Source: cloned_repo\client\src\pages\common\About.jsx
   Content preview: import React from "react";
import {
    Target,
    Users,
    Award,
    Zap,
    Heart,
    Rocket,
    Shield,
    Globe,
    TrendingUp,
    Star,...
   Length: 29949 characters

3. Source: cloned_repo\client\src\pages\candidate\CandidateDashboard.jsx
   Content preview: import React, { useState, useEffect } from "react";
import { Link } from "react-router-dom";
import { toast } from "react-toastify";
// eslint-disable...
   Length: 33232 characters

4. Source: cloned_repo\client\src\components\AdvancedJobSearch.jsx
   Content preview: import React, { useState } from "react";
import {  AnimatePresence } fro

In [4]:
# Test dataset: Modify these queries based on YOUR repository content
test_dataset = [
    {
        "query": "authentication login",
        "relevant_keywords": ["auth", "login", "signin", "password", "token"],
        "relevant_files": ["auth", "login"],
        "description": "Should retrieve authentication and login-related code"
    },
    {
        "query": "database connection",
        "relevant_keywords": ["database", "db", "connection", "mongo", "connect"],
        "relevant_files": ["db", "config", "database"],
        "description": "Should retrieve database configuration code"
    },
    {
        "query": "API routes endpoints",
        "relevant_keywords": ["route", "api", "endpoint", "express", "get", "post"],
        "relevant_files": ["route", "api", "controller"],
        "description": "Should retrieve API routing code"
    },
    {
        "query": "React components",
        "relevant_keywords": ["component", "react", "jsx", "usestate", "props"],
        "relevant_files": ["component", ".jsx"],
        "description": "Should retrieve React component files"
    },
    {
        "query": "resume parsing",
        "relevant_keywords": ["resume", "parse", "parser", "pdf", "extract"],
        "relevant_files": ["resume", "parser"],
        "description": "Should retrieve resume processing code"
    },
    {
        "query": "user schema model",
        "relevant_keywords": ["user", "schema", "model", "mongoose", "candidate"],
        "relevant_files": ["model", "schema", "user", "candidate"],
        "description": "Should retrieve database model definitions"
    },
    {
        "query": "job posting",
        "relevant_keywords": ["job", "post", "create", "vacancy"],
        "relevant_files": ["job", "post"],
        "description": "Should retrieve job posting functionality"
    },
    {
        "query": "application tracking",
        "relevant_keywords": ["application", "track", "status", "applicant"],
        "relevant_files": ["application", "track"],
        "description": "Should retrieve application tracking code"
    }
]

print(f"✅ Created test dataset with {len(test_dataset)} queries")
print("\n📋 Test Queries:")
for i, test in enumerate(test_dataset, 1):
    print(f"{i}. {test['query']} - {test['description']}")

✅ Created test dataset with 8 queries

📋 Test Queries:
1. authentication login - Should retrieve authentication and login-related code
2. database connection - Should retrieve database configuration code
3. API routes endpoints - Should retrieve API routing code
4. React components - Should retrieve React component files
5. resume parsing - Should retrieve resume processing code
6. user schema model - Should retrieve database model definitions
7. job posting - Should retrieve job posting functionality
8. application tracking - Should retrieve application tracking code


In [5]:
# Evaluation metric functions

def is_relevant(doc: Document, relevant_keywords: List[str], relevant_files: List[str]) -> bool:
    """Check if document is relevant based on keywords in content OR file path patterns"""
    content = doc.page_content.lower()
    source = doc.metadata.get('source', '').lower()
    
    keyword_match = any(keyword.lower() in content for keyword in relevant_keywords)
    file_match = any(pattern.lower() in source for pattern in relevant_files)
    
    return keyword_match or file_match


def calculate_precision(retrieved_docs: List[Document], relevant_keywords: List[str], relevant_files: List[str]) -> float:
    """Precision = (Relevant Docs Retrieved) / (Total Docs Retrieved)"""
    if not retrieved_docs:
        return 0.0
    relevant_count = sum(1 for doc in retrieved_docs if is_relevant(doc, relevant_keywords, relevant_files))
    return relevant_count / len(retrieved_docs)


def calculate_recall(retrieved_docs: List[Document], relevant_keywords: List[str], 
                    relevant_files: List[str], vectorstore: Chroma, k_total: int = 20) -> float:
    """Recall = (Relevant Docs Retrieved) / (Total Relevant Docs) - approximated by searching k_total"""
    if not retrieved_docs:
        return 0.0
    
    # Get larger set to estimate total relevant documents
    query = " ".join(relevant_keywords[:3])
    larger_set = vectorstore.similarity_search(query, k=k_total)
    total_relevant = sum(1 for doc in larger_set if is_relevant(doc, relevant_keywords, relevant_files))
    
    if total_relevant == 0:
        return 0.0
    
    relevant_retrieved = sum(1 for doc in retrieved_docs if is_relevant(doc, relevant_keywords, relevant_files))
    return relevant_retrieved / total_relevant


def calculate_reciprocal_rank(retrieved_docs: List[Document], relevant_keywords: List[str], relevant_files: List[str]) -> float:
    """Reciprocal Rank = 1 / (rank of first relevant document)"""
    for rank, doc in enumerate(retrieved_docs, start=1):
        if is_relevant(doc, relevant_keywords, relevant_files):
            return 1.0 / rank
    return 0.0  # No relevant document found


def calculate_mrr(reciprocal_ranks: List[float]) -> float:
    """Mean Reciprocal Rank = Average of all reciprocal ranks"""
    if not reciprocal_ranks:
        return 0.0
    return sum(reciprocal_ranks) / len(reciprocal_ranks)


print("✅ Evaluation functions defined!")

✅ Evaluation functions defined!


In [2]:
# Run evaluation on all test queries
results = []
reciprocal_ranks = []

print("🔍 Running RAG Evaluation...\n")
print("=" * 100)

for i, test in enumerate(test_dataset, 1):
    query = test['query']
    relevant_keywords = test['relevant_keywords']
    relevant_files = test['relevant_files']
    
    # Retrieve documents using RAG system
    retrieved_docs = retriever.get_relevant_documents(query)
    
    # Calculate all metrics
    precision = calculate_precision(retrieved_docs, relevant_keywords, relevant_files)
    recall = calculate_recall(retrieved_docs, relevant_keywords, relevant_files, vectorstore)
    rr = calculate_reciprocal_rank(retrieved_docs, relevant_keywords, relevant_files)
    reciprocal_ranks.append(rr)
    
    # Calculate F1 Score (harmonic mean of precision and recall)
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    results.append({
        'Query': query,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1_score,
        'Reciprocal Rank': rr,
        'Docs Retrieved': len(retrieved_docs)
    })
    
    # Print detailed results for each query
    print(f"\n{i}. Query: '{query}'")
    print(f"   Precision: {precision:.3f} | Recall: {recall:.3f} | F1: {f1_score:.3f} | RR: {rr:.3f}")
    print(f"   Retrieved Documents:")
    
    for j, doc in enumerate(retrieved_docs, 1):
        is_rel = is_relevant(doc, relevant_keywords, relevant_files)
        marker = "✅" if is_rel else "❌"
        source = doc.metadata.get('source', 'Unknown')
        print(f"      {j}. {marker} {source}")

print("\n" + "=" * 100)

# Calculate overall MRR
mrr = calculate_mrr(reciprocal_ranks)
print(f"\n📊 Mean Reciprocal Rank (MRR): {mrr:.3f}")

🔍 Running RAG Evaluation...



NameError: name 'test_dataset' is not defined

In [7]:
# Display results summary
df_results = pd.DataFrame(results)

print("\n📋 DETAILED RESULTS TABLE:\n")
display(df_results)

# Calculate average metrics across all queries
avg_precision = df_results['Precision'].mean()
avg_recall = df_results['Recall'].mean()
avg_f1 = df_results['F1-Score'].mean()

print("\n" + "=" * 60)
print("📊 AVERAGE METRICS ACROSS ALL QUERIES:")
print("=" * 60)
print(f"Average Precision:  {avg_precision:.3f}")
print(f"Average Recall:     {avg_recall:.3f}")
print(f"Average F1-Score:   {avg_f1:.3f}")
print(f"Mean Reciprocal Rank (MRR): {mrr:.3f}")
print("=" * 60)

# Interpretation of results
print("\n💡 INTERPRETATION:")
print(f"\nPrecision ({avg_precision:.1%}): ", end="")
if avg_precision >= 0.7:
    print("✅ Excellent! Most retrieved docs are relevant.")
elif avg_precision >= 0.5:
    print("⚠️ Moderate. Some irrelevant docs are retrieved.")
else:
    print("❌ Low. Many irrelevant docs in results.")

print(f"\nRecall ({avg_recall:.1%}): ", end="")
if avg_recall >= 0.7:
    print("✅ Excellent! Finding most relevant docs.")
elif avg_recall >= 0.5:
    print("⚠️ Moderate. Missing some relevant docs.")
else:
    print("❌ Low. Missing many relevant docs.")

print(f"\nMRR ({mrr:.3f}): ", end="")
if mrr >= 0.7:
    print("✅ Excellent! Relevant docs ranked highly.")
elif mrr >= 0.5:
    print("⚠️ Moderate. Relevant docs not always at top.")
else:
    print("❌ Low. Relevant docs ranked too low.")


📋 DETAILED RESULTS TABLE:



,Query,Precision,Recall,F1-Score,Reciprocal Rank,Docs Retrieved
0,authentication login,1.0,0.300000,0.461538,1.0,6
1,database connection,1.0,0.333333,0.500000,1.0,6
2,API routes endpoints,1.0,0.300000,0.461538,1.0,6
3,React components,1.0,0.300000,0.461538,1.0,6
4,resume parsing,1.0,0.300000,0.461538,1.0,6
5,user schema model,1.0,0.333333,0.500000,1.0,6
6,job posting,1.0,0.315789,0.480000,1.0,6
7,application tracking,1.0,0.300000,0.461538,1.0,6



📊 AVERAGE METRICS ACROSS ALL QUERIES:
Average Precision:  1.000
Average Recall:     0.310
Average F1-Score:   0.473
Mean Reciprocal Rank (MRR): 1.000

💡 INTERPRETATION:

Precision (100.0%): ✅ Excellent! Most retrieved docs are relevant.

Recall (31.0%): ❌ Low. Missing many relevant docs.

MRR (1.000): ✅ Excellent! Relevant docs ranked highly.


In [ ]:
# Create visualizations
fig, axes = plt.subplots(10, 10, figsize=(15, 10))
fig.suptitle('RAG System Evaluation Results', fontsize=16, fontweight='bold')

# Chart 1: Metrics comparison bar chart
metrics_data = {'Metric': ['Precision', 'Recall', 'F1-Score', 'MRR'], 'Score': [avg_precision, avg_recall, avg_f1, mrr]}
ax1 = axes[0, 0]
bars = ax1.bar(metrics_data['Metric'], metrics_data['Score'], color=['#2ecc71', '#3498db', '#e74c3c', '#f39c12'])
ax1.set_ylim(0, 1)
ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('Average Metrics Comparison', fontsize=14, fontweight='bold')
ax1.axhline(y=0.7, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Good threshold')
ax1.legend()
for bar in bars:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height, f'{height:.3f}', ha='center', va='bottom', fontweight='bold')

# Chart 2: Per-query precision and recall
ax2 = axes[0, 1]
x = np.arange(len(df_results))
width = 0.35
ax2.bar(x - width/2, df_results['Precision'], width, label='Precision', color='#2ecc71', alpha=0.8)
ax2.bar(x + width/2, df_results['Recall'], width, label='Recall', color='#3498db', alpha=0.8)
ax2.set_xlabel('Query Number', fontsize=12)
ax2.set_ylabel('Score', fontsize=12)
ax2.set_title('Precision & Recall per Query', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels([f'Q{i+1}' for i in range(len(df_results))])
ax2.legend()
ax2.set_ylim(0, 1.1)

# Chart 3: Reciprocal Rank distribution
ax3 = axes[1, 0]
ax3.plot(range(1, len(reciprocal_ranks) + 1), reciprocal_ranks, marker='o', linewidth=2, markersize=8, color='#f39c12')
ax3.axhline(y=mrr, color='red', linestyle='--', linewidth=2, label=f'MRR = {mrr:.3f}')
ax3.set_xlabel('Query Number', fontsize=12)
ax3.set_ylabel('Reciprocal Rank', fontsize=12)
ax3.set_title('Reciprocal Rank per Query', fontsize=14, fontweight='bold')
ax3.legend()
ax3.set_ylim(0, 1.1)
ax3.grid(True, alpha=0.3)

# Chart 4: F1-Score distribution
ax4 = axes[1, 1]
queries_short = [f"Q{i+1}" for i in range(len(df_results))]
colors = plt.cm.RdYlGn(df_results['F1-Score'])
bars = ax4.barh(queries_short, df_results['F1-Score'], color=colors)
ax4.set_xlabel('F1-Score', fontsize=12)
ax4.set_title('F1-Score Distribution', fontsize=14, fontweight='bold')
ax4.set_xlim(0, 1)
for i, (bar, score) in enumerate(zip(bars, df_results['F1-Score'])):
    ax4.text(score + 0.02, bar.get_y() + bar.get_height()/2, f'{score:.3f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ Visualizations generated!")

In [1]:
# Aggregate statistics across all queries
total_retrieved = sum(res['Docs Retrieved'] for res in results)
total_relevant_retrieved = 0
total_irrelevant_retrieved = 0

for test in test_dataset:
    retrieved_docs = retriever.get_relevant_documents(test['query'])
    for doc in retrieved_docs:
        if is_relevant(doc, test['relevant_keywords'], test['relevant_files']):
            total_relevant_retrieved += 1
        else:
            total_irrelevant_retrieved += 1

print("\n" + "=" * 60)
print("📊 AGGREGATE STATISTICS:")
print("=" * 60)
print(f"Total Queries Tested:          {len(test_dataset)}")
print(f"Total Documents Retrieved:     {total_retrieved}")
print(f"Relevant Documents Retrieved:  {total_relevant_retrieved}")
print(f"Irrelevant Documents Retrieved: {total_irrelevant_retrieved}")
print(f"\nOverall Precision: {total_relevant_retrieved / total_retrieved:.3f}")
print("=" * 60)

NameError: name 'results' is not defined

In [ ]:
# Export results to CSV files
output_file = "rag_evaluation_results.csv"
df_results.to_csv(output_file, index=False)
print(f"✅ Results saved to: {output_file}")

# Export summary statistics
summary = {
    'Metric': ['Average Precision', 'Average Recall', 'Average F1-Score', 'Mean Reciprocal Rank (MRR)'],
    'Value': [avg_precision, avg_recall, avg_f1, mrr]
}
df_summary = pd.DataFrame(summary)
summary_file = "rag_evaluation_summary.csv"
df_summary.to_csv(summary_file, index=False)
print(f"✅ Summary saved to: {summary_file}")

print("\n🎉 Evaluation Complete!")

In [ ]:
# Generate recommendations based on metrics
recommendations = []

if avg_precision < 0.6:
    recommendations.append("⚠️ LOW PRECISION: Improve chunking or use better embeddings (e.g., CodeBERT)")
if avg_recall < 0.6:
    recommendations.append("⚠️ LOW RECALL: Increase k or implement query expansion")
if mrr < 0.6:
    recommendations.append("⚠️ LOW MRR: Implement re-ranking with cross-encoders")
if avg_f1 < 0.6:
    recommendations.append("⚠️ LOW F1-SCORE: Use hybrid search (semantic + keyword)")
if not recommendations:
    recommendations.append("✅ EXCELLENT PERFORMANCE! Your RAG system is working well.")

print("\n" + "=" * 80)
print("💡 RECOMMENDATIONS FOR IMPROVEMENT:")
print("=" * 80)
for i, rec in enumerate(recommendations, 1):
    print(f"\n{i}. {rec}")

print("\n" + "=" * 80)
print("\n📚 Additional Optimization Strategies:")
print("   • Use domain-specific embeddings (e.g., CodeBERT for code)")
print("   • Implement query expansion with synonyms")
print("   • Add metadata filtering (file type, language)")
print("   • Use hybrid search (BM25 + semantic)")
print("   • Implement re-ranking with cross-encoders")
print("   • Adjust chunk size and overlap parameters")
print("   • Fine-tune retrieval k parameter based on query type")